# Transfer Learning GAN - Leave-One-LMI-Out 3-Fold

Setiap fold menahan 1 dari 3 LMI asli sebagai test LMI. Dua LMI asli lain dipakai sebagai reference generator untuk membuat LMI train/validation.

## 1. Import dan Konfigurasi

In [8]:
import os
import json
import random
import warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_curve,
    auc,
    average_precision_score,
    precision_recall_curve,
)
from sklearn.preprocessing import label_binarize
from scipy.signal import butter, filtfilt


In [9]:
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebook' else Path('.').resolve()
DATA_READY_DIR = PROJECT_ROOT / 'data_ready_ptb'
OUTPUT_DIR = PROJECT_ROOT / 'crossval_comparison' / 'localization_leave_one_lmi_3fold_gan'
PRETRAIN_PATH = PROJECT_ROOT / 'pretrain_model_localization' / 'best_model_localization_evidence.pth'
PTB_RAW_DIR = PROJECT_ROOT.parent.parent / 'ptb-diagnostic-ecg-database'

class_names = ["NORM", "IMI", "AMI", "LMI"]
label_to_idx = {name: i for i, name in enumerate(class_names)}
LMI_IDX = label_to_idx['LMI']
lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

RAW_FS = 1000
TARGET_FS = 100
TARGET_LEN = 1000
N_LEADS = 12
SEGMENT_SECONDS = 10
RAW_SEGMENT_LEN = RAW_FS * SEGMENT_SECONDS
BANDPASS_LOWCUT = 0.5
BANDPASS_HIGHCUT = 40.0
BANDPASS_ORDER = 4

CONFIG = {
    'seed': 42,
    'batch_size': 32,
    'epochs': 100,
    'use_early_stopping': False,
    'patience': None,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'gradient_clip_norm': 2.0,
    'focal_gamma': 1.5,
    'use_weighted_sampler': True,
    'selection_metric': 'macro_f1',
    'lmi_train_total': 24,
    'lmi_val_total': 6,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

MODEL_CONFIGS = [
    {'model_name': 'localization_no_pretrain', 'pretrain_path': None, 'freeze_mode': 'full', 'learning_rate': 1e-3},
    {'model_name': 'localization_frozen_backbone', 'pretrain_path': PRETRAIN_PATH, 'freeze_mode': 'frozen_backbone', 'learning_rate': 1e-3},
    {'model_name': 'localization_partial_finetune', 'pretrain_path': PRETRAIN_PATH, 'freeze_mode': 'partial', 'learning_rate': 5e-4},
    {'model_name': 'localization_full_finetune', 'pretrain_path': PRETRAIN_PATH, 'freeze_mode': 'full', 'learning_rate': 1e-4},
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=24):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
device = torch.device(CONFIG['device'])

print(json.dumps(CONFIG, indent=2))
print('DATA_READY_DIR:', DATA_READY_DIR)
print('PTB_RAW_DIR    :', PTB_RAW_DIR)
print('OUTPUT_DIR     :', OUTPUT_DIR)
print('PRETRAIN_PATH  :', PRETRAIN_PATH)
print('Device:', device)


{
  "seed": 42,
  "batch_size": 32,
  "epochs": 100,
  "use_early_stopping": false,
  "patience": null,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "gradient_clip_norm": 2.0,
  "focal_gamma": 1.5,
  "use_weighted_sampler": true,
  "selection_metric": "macro_f1",
  "lmi_train_total": 24,
  "lmi_val_total": 6,
  "device": "cuda"
}
DATA_READY_DIR: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-classification/data_ready_ptb
PTB_RAW_DIR    : /home/nugee/code-program/code-thesis/ptb-diagnostic-ecg-database
OUTPUT_DIR     : /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-classification/crossval_comparison/localization_leave_one_lmi_3fold_gan
PRETRAIN_PATH  : /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-classification/pretrain_model_localization/best_model_localization_evidence.pth
Device: cuda


## 2. Load Data Ready PTB dan 3 Raw LMI

In [10]:
def load_ready_split(split):
    x = np.load(DATA_READY_DIR / f'x_{split}.npy').astype(np.float32)
    y = np.load(DATA_READY_DIR / f'y_{split}.npy')
    if y.ndim > 1 and y.shape[1] > 1:
        y_idx = y.argmax(axis=1).astype(np.int64)
    else:
        y_idx = y.reshape(-1).astype(np.int64)
    return x, y_idx

def validate_ecg_array(x, name):
    if x.ndim != 3:
        raise ValueError(f'{name}: expected [N, length, leads], got {x.shape}')
    if x.shape[1:] != (TARGET_LEN, N_LEADS):
        raise ValueError(f'{name}: expected [N, {TARGET_LEN}, {N_LEADS}], got {x.shape}')
    if not np.isfinite(x).all():
        raise ValueError(f'{name}: contains NaN or Inf')

x_train_old, y_train_old = load_ready_split('train')
x_val_old, y_val_old = load_ready_split('val')
x_test_old, y_test_old = load_ready_split('test')
for split_name, x in [('train', x_train_old), ('val', x_val_old), ('test', x_test_old)]:
    validate_ecg_array(x, split_name)

# Non-LMI base follows the previous GAN protocol: old LMI is removed from all ready splits.
train_non_lmi = y_train_old != LMI_IDX
val_non_lmi = y_val_old != LMI_IDX
test_non_lmi = y_test_old != LMI_IDX
x_train_base, y_train_base = x_train_old[train_non_lmi], y_train_old[train_non_lmi]
x_val_base, y_val_base = x_val_old[val_non_lmi], y_val_old[val_non_lmi]
x_test_base, y_test_base = x_test_old[test_non_lmi], y_test_old[test_non_lmi]

meta = pd.read_csv(PTB_RAW_DIR / 'meta.csv')
ptb_raw = np.load(PTB_RAW_DIR / 'data_raw.npz')

acute = meta['Acute_infarction_(localization)'].fillna('').str.strip().str.lower()
former = meta['Former_infarction_(localization)'].fillna('').str.strip().str.lower()
no_acute = acute.isin(['', 'no', 'unknown'])
no_former = former.isin(['', 'no', 'unknown'])
lmi_mask = ((acute == 'lateral') & no_former) | ((former == 'lateral') & no_acute)
lmi_meta = meta.loc[lmi_mask].copy().reset_index(drop=True)
assert len(lmi_meta) == 3, f'Expected exactly 3 pure LMI records, got {len(lmi_meta)}'
display(lmi_meta[['patient', 'record_id', 'fs', 'sig_len', 'Acute_infarction_(localization)', 'Former_infarction_(localization)']])

print('Base non-LMI train:', x_train_base.shape, np.bincount(y_train_base, minlength=len(class_names)))
print('Base non-LMI val  :', x_val_base.shape, np.bincount(y_val_base, minlength=len(class_names)))
print('Base non-LMI test :', x_test_base.shape, np.bincount(y_test_base, minlength=len(class_names)))


,patient,record_id,fs,sig_len,Acute_infarction_(localization),Former_infarction_(localization)
0,patient043,s0141lre,1000,115200,lateral,no
1,patient043,s0144lre,1000,115200,lateral,no
2,patient043,s0278lre,1000,115200,lateral,no


Base non-LMI train: (141, 1000, 12) [56 56 29  0]
Base non-LMI val  : (20, 1000, 12) [8 8 4 0]
Base non-LMI test : (41, 1000, 12) [16 16  9  0]


## 3. Generator LMI Per Fold

In [11]:
def butterworth_bandpass_filter(data, fs, lowcut=0.5, highcut=40.0, order=4, axis=0):
    nyquist = 0.5 * fs
    b, a = butter(order, [lowcut / nyquist, highcut / nyquist], btype='band')
    return filtfilt(b, a, data, axis=axis)

def preprocess_raw_to_ready(raw_signal_1000hz):
    signal = raw_signal_1000hz[:, :N_LEADS].astype(np.float32)
    factor = RAW_FS // TARGET_FS
    signal_100hz = signal[::factor]
    if signal_100hz.shape[0] < TARGET_LEN:
        padded = np.zeros((TARGET_LEN, signal_100hz.shape[1]), dtype=np.float32)
        padded[:signal_100hz.shape[0]] = signal_100hz
        signal_100hz = padded
    else:
        signal_100hz = signal_100hz[:TARGET_LEN]
    filtered = butterworth_bandpass_filter(
        signal_100hz,
        fs=TARGET_FS,
        lowcut=BANDPASS_LOWCUT,
        highcut=BANDPASS_HIGHCUT,
        order=BANDPASS_ORDER,
        axis=0,
    )
    return filtered.astype(np.float32)

def extract_random_raw_window(raw, rng):
    if raw.shape[0] <= RAW_SEGMENT_LEN:
        start = 0
    else:
        start = int(rng.integers(0, raw.shape[0] - RAW_SEGMENT_LEN + 1))
    return raw[start:start + RAW_SEGMENT_LEN, :N_LEADS].copy().astype(np.float32), start

def raw_amplitude_scale(x, rng, low=0.98, high=1.02):
    scale = rng.uniform(low, high, size=(1, x.shape[1])).astype(np.float32)
    return x * scale

def raw_time_shift(x, rng, max_shift=50):
    shift = int(rng.integers(-max_shift, max_shift + 1))
    return np.roll(x, shift, axis=0)

def raw_jitter(x, rng, noise_ratio=0.004):
    lead_std = x.std(axis=0, keepdims=True) + 1e-8
    noise = rng.normal(0, noise_ratio, size=x.shape).astype(np.float32) * lead_std
    return x + noise

def raw_baseline_wander(x, rng, fs=1000, max_amp_ratio=0.006):
    t = np.arange(x.shape[0]) / fs
    freq = rng.uniform(0.05, 0.20)
    phase = rng.uniform(0, 2 * np.pi)
    lead_std = x.std(axis=0, keepdims=True) + 1e-8
    amp = rng.uniform(0.0, max_amp_ratio, size=(1, x.shape[1])).astype(np.float32) * lead_std
    return x + np.sin(2 * np.pi * freq * t[:, None] + phase).astype(np.float32) * amp

def generate_from_references(raw_refs, n_samples, seed):
    rng = np.random.default_rng(seed)
    synthetic = []
    rows = []
    for i in range(n_samples):
        ref_idx = int(rng.integers(0, len(raw_refs)))
        x, start = extract_random_raw_window(raw_refs[ref_idx], rng)
        x = raw_amplitude_scale(x, rng)
        x = raw_time_shift(x, rng)
        x = raw_baseline_wander(x, rng, fs=RAW_FS)
        x = raw_jitter(x, rng)
        synthetic.append(preprocess_raw_to_ready(x))
        rows.append({'sample_id': f'lmi_aug_{i + 1:03d}', 'source_reference_index': ref_idx, 'raw_start_sample': start})
    return np.stack(synthetic).astype(np.float32), pd.DataFrame(rows)

def get_lmi_raw(row):
    key = f"{row['patient']}/{row['record_id']}"
    return ptb_raw[key][:, :N_LEADS].astype(np.float32)

def build_leave_one_lmi_split(test_lmi_index, seed):
    generator_indices = [idx for idx in range(len(lmi_meta)) if idx != test_lmi_index]
    raw_refs = [get_lmi_raw(lmi_meta.iloc[idx]) for idx in generator_indices]
    n_lmi_total = CONFIG['lmi_train_total'] + CONFIG['lmi_val_total']
    x_lmi_generated, gen_meta = generate_from_references(raw_refs, n_lmi_total, seed=seed)
    y_lmi_generated = np.full(n_lmi_total, LMI_IDX, dtype=np.int64)

    x_train = np.concatenate([x_train_base, x_lmi_generated[:CONFIG['lmi_train_total']]], axis=0).astype(np.float32)
    y_train = np.concatenate([y_train_base, y_lmi_generated[:CONFIG['lmi_train_total']]], axis=0).astype(np.int64)
    x_val = np.concatenate([x_val_base, x_lmi_generated[CONFIG['lmi_train_total']:]], axis=0).astype(np.float32)
    y_val = np.concatenate([y_val_base, y_lmi_generated[CONFIG['lmi_train_total']:]], axis=0).astype(np.int64)

    x_lmi_test = preprocess_raw_to_ready(get_lmi_raw(lmi_meta.iloc[test_lmi_index]))[None, ...]
    y_lmi_test = np.array([LMI_IDX], dtype=np.int64)
    x_test = np.concatenate([x_test_base, x_lmi_test], axis=0).astype(np.float32)
    y_test = np.concatenate([y_test_base, y_lmi_test], axis=0).astype(np.int64)

    rng = np.random.default_rng(seed)
    train_perm = rng.permutation(len(y_train))
    val_perm = rng.permutation(len(y_val))
    x_train, y_train = x_train[train_perm], y_train[train_perm]
    x_val, y_val = x_val[val_perm], y_val[val_perm]

    fold_meta = {
        'test_lmi_index': test_lmi_index,
        'test_lmi_record': f"{lmi_meta.loc[test_lmi_index, 'patient']}/{lmi_meta.loc[test_lmi_index, 'record_id']}",
        'generator_lmi_indices': generator_indices,
        'generator_lmi_records': [f"{lmi_meta.loc[idx, 'patient']}/{lmi_meta.loc[idx, 'record_id']}" for idx in generator_indices],
    }
    return x_train, y_train, x_val, y_val, x_test, y_test, fold_meta, gen_meta


## 4. Dataset, Normalisasi, dan Arsitektur

In [12]:
class PerLeadZScore:
    def __init__(self, eps=1e-6):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, x):
        self.mean_ = x.mean(axis=(0, 1), keepdims=True)
        self.std_ = np.maximum(x.std(axis=(0, 1), keepdims=True), self.eps)
        return self

    def transform(self, x):
        if self.mean_ is None or self.std_ is None:
            raise RuntimeError('Normalizer must be fit on the current train fold first.')
        return ((x - self.mean_) / self.std_).astype(np.float32)

class ECGDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

class SE1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(4, channels // reduction)
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.fc1 = nn.Conv1d(channels, hidden, kernel_size=1)
        self.fc2 = nn.Conv1d(hidden, channels, kernel_size=1)

    def forward(self, x):
        w = self.gap(x)
        w = F.relu(self.fc1(w))
        w = torch.sigmoid(self.fc2(w))
        return x * w

class LeadWiseAttention(nn.Module):
    def __init__(self, n_leads=12, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        # x: [B, L, C]. Feature per lead: mean, std, mean absolute amplitude.
        mean = x.mean(dim=1)
        std = x.std(dim=1)
        abs_mean = x.abs().mean(dim=1)
        lead_features = torch.stack([mean, std, abs_mean], dim=-1)  # [B, C, 3]
        logits = self.net(lead_features).squeeze(-1)  # [B, C]
        weights = F.softmax(logits, dim=1)
        x_weighted = x * weights.unsqueeze(1)
        return x_weighted, weights

class TemporalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.W = nn.Parameter(torch.randn(dim, dim) * 0.02)
        self.u = nn.Parameter(torch.randn(dim) * 0.02)

    def forward(self, x):
        # x: [B, T, D]
        scores = torch.tanh(x @ self.W) @ self.u
        weights = F.softmax(scores, dim=1)
        context = (x * weights.unsqueeze(-1)).sum(dim=1)
        return context, weights

class CNNLSTMSEAttentionLocalization(nn.Module):
    def __init__(self, in_leads=12, num_classes=4, dropout=0.35):
        super().__init__()
        self.in_leads = in_leads
        self.lead_attention = LeadWiseAttention(n_leads=in_leads)

        self.conv1 = nn.Conv1d(in_leads, 64, kernel_size=7, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.se1 = SE1D(64)
        self.pool1 = nn.MaxPool1d(2)

        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.se2 = SE1D(128)
        self.pool2 = nn.MaxPool1d(2)

        self.conv3 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.se3 = SE1D(128)
        self.pool3 = nn.MaxPool1d(2)

        self.conv4 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm1d(256)
        self.se4 = SE1D(256)

        self.lstm1 = nn.LSTM(input_size=256, hidden_size=128, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(input_size=256, hidden_size=96, batch_first=True, bidirectional=True)
        self.temporal_attention = TemporalAttention(192)
        self.timedense = nn.Linear(192, 128)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(128, num_classes)

    def forward_features(self, x):
        # Accept [B, L, C].
        x_weighted, lead_weights = self.lead_attention(x)
        x_c = x_weighted.permute(0, 2, 1)  # [B, C, L]

        x_c = self.pool1(self.se1(F.relu(self.bn1(self.conv1(x_c)))))
        x_c = self.pool2(self.se2(F.relu(self.bn2(self.conv2(x_c)))))
        x_c = self.pool3(self.se3(F.relu(self.bn3(self.conv3(x_c)))))
        x_c = self.se4(F.relu(self.bn4(self.conv4(x_c))))

        x_t = x_c.permute(0, 2, 1)
        x_t, _ = self.lstm1(x_t)
        x_t, _ = self.lstm2(x_t)
        context, temporal_weights = self.temporal_attention(x_t)
        features = self.dropout(F.relu(self.timedense(context)))
        return features, lead_weights, temporal_weights

    def forward(self, x, return_attention=False):
        features, lead_weights, temporal_weights = self.forward_features(x)
        logits = self.classifier(features)
        if return_attention:
            return logits, {'lead_attention': lead_weights, 'temporal_attention': temporal_weights}
        return logits

class FocalLoss(nn.Module):
    def __init__(self, gamma=1.5, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, target):
        ce = F.cross_entropy(logits, target, reduction='none', weight=self.weight)
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


## 5. Helper Training dan Evaluasi

In [13]:
def make_loaders(x_train, y_train, x_val, y_val, x_test, y_test, batch_size):
    normalizer = PerLeadZScore().fit(x_train)
    x_train_n = normalizer.transform(x_train)
    x_val_n = normalizer.transform(x_val)
    x_test_n = normalizer.transform(x_test)

    train_ds = ECGDataset(x_train_n, y_train)
    val_ds = ECGDataset(x_val_n, y_val)
    test_ds = ECGDataset(x_test_n, y_test)

    if CONFIG['use_weighted_sampler']:
        counts = np.bincount(y_train, minlength=len(class_names))
        weights = 1.0 / np.maximum(counts, 1)
        sample_weights = weights[y_train]
        sampler = WeightedRandomSampler(
            weights=torch.tensor(sample_weights, dtype=torch.double),
            num_samples=len(sample_weights),
            replacement=True,
        )
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader, normalizer

def load_localization_pretrain(model, pretrain_path):
    if pretrain_path is None:
        return {'loaded': False, 'missing_keys': [], 'unexpected_keys': [], 'load_mode': 'no_pretrain'}

    pretrain_path = Path(pretrain_path)
    if not pretrain_path.exists():
        warnings.warn(f'Pretrain checkpoint not found: {pretrain_path}. Model starts from random initialization.', RuntimeWarning)
        return {'loaded': False, 'missing_keys': [], 'unexpected_keys': [], 'load_mode': 'missing'}

    try:
        checkpoint = torch.load(pretrain_path, map_location='cpu', weights_only=True)
        load_mode = 'weights_only=True'
    except Exception as exc:
        # PyTorch 2.6 defaults to weights_only=True. The local localization checkpoint
        # stores numpy metadata, so a trusted local checkpoint may need full pickle load.
        warnings.warn(
            f'weights_only=True failed for local checkpoint {pretrain_path.name}: {exc}. '
            'Retrying with weights_only=False because this checkpoint is produced by this project.',
            RuntimeWarning,
        )
        checkpoint = torch.load(pretrain_path, map_location='cpu', weights_only=False)
        load_mode = 'weights_only=False'

    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint

    result = model.load_state_dict(state_dict, strict=False)
    return {
        'loaded': True,
        'missing_keys': list(result.missing_keys),
        'unexpected_keys': list(result.unexpected_keys),
        'load_mode': load_mode,
    }

def configure_trainable_layers(model, freeze_mode):
    for p in model.parameters():
        p.requires_grad = True

    if freeze_mode == 'full':
        return

    if freeze_mode == 'frozen_backbone':
        for name, p in model.named_parameters():
            p.requires_grad = name.startswith('classifier')
        return

    if freeze_mode == 'partial':
        trainable_keywords = ('conv4', 'bn4', 'se4', 'lstm2', 'temporal_attention', 'timedense', 'classifier')
        for name, p in model.named_parameters():
            p.requires_grad = any(key in name for key in trainable_keywords)
        return

    raise ValueError(f'Unknown freeze_mode: {freeze_mode}')

def count_trainable_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, total_correct, total_count = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip_norm'])
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_count += xb.size(0)
    return total_loss / max(total_count, 1), total_correct / max(total_count, 1)



def safe_balanced_accuracy(y_true, y_pred, labels=None):
    if labels is None:
        labels = list(range(len(class_names)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    support = cm.sum(axis=1)
    present = support > 0
    if not np.any(present):
        return np.nan
    recalls = np.divide(
        np.diag(cm),
        support,
        out=np.zeros_like(support, dtype=float),
        where=support > 0,
    )
    return float(recalls[present].mean())

def evaluate_model(model, loader, criterion=None):
    model.eval()
    total_loss, total_count = 0.0, 0
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            if criterion is not None:
                loss = criterion(logits, yb)
                total_loss += loss.item() * xb.size(0)
                total_count += xb.size(0)
            prob = torch.softmax(logits, dim=1)
            y_true.append(yb.cpu().numpy())
            y_pred.append(prob.argmax(dim=1).cpu().numpy())
            y_prob.append(prob.cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    y_prob = np.vstack(y_prob)
    summary = {
        'loss': total_loss / max(total_count, 1) if criterion is not None else np.nan,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': safe_balanced_accuracy(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }
    return summary, y_true, y_pred, y_prob

def plot_training_curve(history, title, save_path):
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
    axes[0].plot(df['epoch'], df['train_loss'], label='Train')
    axes[0].plot(df['epoch'], df['val_loss'], label='Val', linestyle='--')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(df['epoch'], df['train_acc'], label='Train')
    axes[1].plot(df['epoch'], df['val_acc'], label='Val', linestyle='--')
    axes[1].plot(df['epoch'], df['val_macro_f1'], label='Val Macro F1', linestyle=':')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Score')
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def plot_confusion_matrix(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=(7, 6), dpi=150)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    return cm

def plot_roc_pr_curves(y_true, y_prob, title_prefix, save_path):
    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

    for i, name in enumerate(class_names):
        if y_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        precision, recall, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
        ap = average_precision_score(y_bin[:, i], y_prob[:, i])
        axes[0].plot(fpr, tpr, label=f'{name} AUC={roc_auc:.3f}')
        axes[1].plot(recall, precision, label=f'{name} AP={ap:.3f}')

    axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
    axes[0].set_title(f'{title_prefix} ROC')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].grid(alpha=0.3)
    axes[0].legend(fontsize=8)

    axes[1].set_title(f'{title_prefix} Precision-Recall')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].grid(alpha=0.3)
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def classification_report_df(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        labels=list(range(len(class_names))),
        digits=4,
        zero_division=0,
        output_dict=True,
    )
    return pd.DataFrame(report).T


## 6. Jalankan Leave-One-LMI-Out 3-Fold

In [14]:
all_fold_summaries = []
all_histories = {}
fold_protocol_rows = []

for model_cfg in MODEL_CONFIGS:
    model_name = model_cfg['model_name']
    model_dir = OUTPUT_DIR / model_name
    (model_dir / 'fold_models').mkdir(parents=True, exist_ok=True)
    (model_dir / 'fold_histories').mkdir(parents=True, exist_ok=True)
    (model_dir / 'fold_plots').mkdir(parents=True, exist_ok=True)
    (model_dir / 'fold_reports').mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 90}\nMODEL: {model_name}\n{'=' * 90}")

    for fold, test_lmi_index in enumerate(range(3), start=1):
        set_seed(CONFIG['seed'] + fold)
        fold_seed = CONFIG['seed'] + (100 * fold)
        fold_dir = model_dir / 'fold_plots' / f'fold{fold}_test_lmi{test_lmi_index}'
        fold_dir.mkdir(parents=True, exist_ok=True)

        x_train, y_train, x_val, y_val, x_test, y_test, fold_meta, gen_meta = build_leave_one_lmi_split(
            test_lmi_index=test_lmi_index,
            seed=fold_seed,
        )
        fold_protocol_rows.append({'model_name': model_name, 'fold': fold, **fold_meta})
        gen_meta.to_csv(model_dir / 'fold_reports' / f'fold{fold}_generated_lmi_metadata.csv', index=False)

        train_loader, val_loader, test_loader, normalizer = make_loaders(
            x_train, y_train, x_val, y_val, x_test, y_test, CONFIG['batch_size']
        )

        class_counts = np.bincount(y_train, minlength=len(class_names))
        class_weights_np = class_counts.sum() / np.maximum(class_counts, 1)
        class_weights_np = class_weights_np / class_weights_np.mean()
        class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)

        model = CNNLSTMSEAttentionLocalization(in_leads=len(lead_names), num_classes=len(class_names)).to(device)
        load_info = load_localization_pretrain(model, model_cfg['pretrain_path'])
        configure_trainable_layers(model, model_cfg['freeze_mode'])
        total_params, trainable_params = count_trainable_parameters(model)

        criterion = FocalLoss(gamma=CONFIG['focal_gamma'], weight=class_weights)
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=model_cfg.get('learning_rate', CONFIG['learning_rate']),
            weight_decay=CONFIG['weight_decay'],
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

        best_score = -np.inf
        best_epoch = 0
        best_state = None
        epochs_without_improvement = 0
        history = []

        print(
            f"\nFold {fold}/3 | test_lmi_index={test_lmi_index} | "
            f"test={fold_meta['test_lmi_record']} | generator={fold_meta['generator_lmi_records']}"
        )
        print('train:', x_train.shape, np.bincount(y_train, minlength=len(class_names)))
        print('val  :', x_val.shape, np.bincount(y_val, minlength=len(class_names)))
        print('test :', x_test.shape, np.bincount(y_test, minlength=len(class_names)))

        for epoch in range(1, CONFIG['epochs'] + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
            val_summary, _, _, _ = evaluate_model(model, val_loader, criterion)
            selection_score = val_summary[CONFIG['selection_metric']]
            scheduler.step(selection_score)

            row = {
                'model_name': model_name,
                'fold': fold,
                'test_lmi_index': test_lmi_index,
                'epoch': epoch,
                'train_loss': train_loss,
                'train_acc': train_acc,
                'val_loss': val_summary['loss'],
                'val_acc': val_summary['accuracy'],
                'val_balanced_accuracy': val_summary['balanced_accuracy'],
                'val_macro_f1': val_summary['macro_f1'],
                'val_weighted_f1': val_summary['weighted_f1'],
                'learning_rate': optimizer.param_groups[0]['lr'],
            }
            history.append(row)

            if selection_score > best_score:
                best_score = selection_score
                best_epoch = epoch
                best_state = deepcopy(model.state_dict())
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if epoch == 1 or epoch % 5 == 0 or epoch == CONFIG['epochs']:
                print(
                    f"Epoch {epoch:03d} | train_loss={train_loss:.4f} val_loss={val_summary['loss']:.4f} "
                    f"val_acc={val_summary['accuracy']:.4f} val_macro_f1={val_summary['macro_f1']:.4f}"
                )

            if CONFIG.get('use_early_stopping', False) and epochs_without_improvement >= CONFIG['patience']:
                print(f"Early stopping at epoch {epoch}; best epoch {best_epoch} ({CONFIG['selection_metric']}={best_score:.4f})")
                break

        model.load_state_dict(best_state)
        hist_df = pd.DataFrame(history)
        hist_path = model_dir / 'fold_histories' / f'fold{fold}_history.csv'
        hist_df.to_csv(hist_path, index=False)
        all_histories[f'{model_name}_fold{fold}'] = hist_df

        ckpt_path = model_dir / 'fold_models' / f'fold{fold}_test_lmi{test_lmi_index}_best_epoch{best_epoch}_{CONFIG["selection_metric"]}{best_score:.4f}.pth'
        torch.save({
            'model_state_dict': best_state,
            'model_name': model_name,
            'fold': fold,
            'test_lmi_index': test_lmi_index,
            'fold_meta': fold_meta,
            'best_epoch': best_epoch,
            'best_score': best_score,
            'class_names': class_names,
            'lead_names': lead_names,
            'normalizer_mean': normalizer.mean_,
            'normalizer_std': normalizer.std_,
            'config': CONFIG,
            'model_config': model_cfg,
            'pretrain_load_info': load_info,
        }, ckpt_path)

        plot_training_curve(hist_df, f'{model_name} - Fold {fold} Test LMI {test_lmi_index}', fold_dir / 'training_curve.png')

        val_summary, y_val_true, y_val_pred, y_val_prob = evaluate_model(model, val_loader, criterion)
        test_summary, y_test_true, y_test_pred, y_test_prob = evaluate_model(model, test_loader, criterion)

        val_cm = plot_confusion_matrix(y_val_true, y_val_pred, f'{model_name} Fold {fold} Validation', fold_dir / 'confusion_matrix_validation.png')
        test_cm = plot_confusion_matrix(y_test_true, y_test_pred, f'{model_name} Fold {fold} Test', fold_dir / 'confusion_matrix_test.png')
        plot_roc_pr_curves(y_val_true, y_val_prob, f'{model_name} Fold {fold} Validation', fold_dir / 'roc_pr_validation.png')
        plot_roc_pr_curves(y_test_true, y_test_prob, f'{model_name} Fold {fold} Test', fold_dir / 'roc_pr_test.png')

        classification_report_df(y_val_true, y_val_pred).to_csv(model_dir / 'fold_reports' / f'fold{fold}_validation_classification_report.csv')
        classification_report_df(y_test_true, y_test_pred).to_csv(model_dir / 'fold_reports' / f'fold{fold}_test_classification_report.csv')
        pd.DataFrame(val_cm, index=class_names, columns=class_names).to_csv(model_dir / 'fold_reports' / f'fold{fold}_validation_confusion_matrix.csv')
        pd.DataFrame(test_cm, index=class_names, columns=class_names).to_csv(model_dir / 'fold_reports' / f'fold{fold}_test_confusion_matrix.csv')

        summary_row = {
            'model_name': model_name,
            'fold': fold,
            'test_lmi_index': test_lmi_index,
            'test_lmi_record': fold_meta['test_lmi_record'],
            'generator_lmi_records': ';'.join(fold_meta['generator_lmi_records']),
            'best_epoch': best_epoch,
            'best_selection_metric': CONFIG['selection_metric'],
            'best_selection_score': best_score,
            'pretrained_loaded': load_info['loaded'],
            'freeze_mode': model_cfg['freeze_mode'],
            'train_samples': len(y_train),
            'val_samples': len(y_val),
            'test_samples': len(y_test),
            'trainable_params': trainable_params,
            'total_params': total_params,
            'checkpoint_path': str(ckpt_path),
            'val_accuracy': val_summary['accuracy'],
            'val_balanced_accuracy': val_summary['balanced_accuracy'],
            'val_macro_f1': val_summary['macro_f1'],
            'val_weighted_f1': val_summary['weighted_f1'],
            'test_accuracy': test_summary['accuracy'],
            'test_balanced_accuracy': test_summary['balanced_accuracy'],
            'test_macro_f1': test_summary['macro_f1'],
            'test_weighted_f1': test_summary['weighted_f1'],
        }
        all_fold_summaries.append(summary_row)
        pd.DataFrame([summary_row]).to_csv(model_dir / 'fold_reports' / f'fold{fold}_summary.csv', index=False)
        print(f"Fold {fold} done | val_macro_f1={val_summary['macro_f1']:.4f} test_macro_f1={test_summary['macro_f1']:.4f}")

summary_df = pd.DataFrame(all_fold_summaries)
summary_df.to_csv(OUTPUT_DIR / 'summary_per_model_per_fold.csv', index=False)
pd.DataFrame(fold_protocol_rows).to_csv(OUTPUT_DIR / 'leave_one_lmi_protocol.csv', index=False)
summary_df.groupby('model_name').agg({
    'val_accuracy': ['mean', 'std'],
    'val_macro_f1': ['mean', 'std'],
    'test_accuracy': ['mean', 'std'],
    'test_macro_f1': ['mean', 'std'],
}).to_csv(OUTPUT_DIR / 'summary_by_model.csv')

with pd.ExcelWriter(OUTPUT_DIR / 'crossval_summary.xlsx', engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='summary_per_fold', index=False)
    summary_df.groupby('model_name').agg({
        'val_accuracy': ['mean', 'std'],
        'val_macro_f1': ['mean', 'std'],
        'test_accuracy': ['mean', 'std'],
        'test_macro_f1': ['mean', 'std'],
    }).to_excel(writer, sheet_name='summary_by_model')
    pd.DataFrame(fold_protocol_rows).to_excel(writer, sheet_name='leave_one_lmi_protocol', index=False)
    for sheet_name, hist_df in all_histories.items():
        hist_df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

print('Saved summary:', OUTPUT_DIR / 'summary_per_model_per_fold.csv')
print('Saved protocol:', OUTPUT_DIR / 'leave_one_lmi_protocol.csv')
print('Saved Excel  :', OUTPUT_DIR / 'crossval_summary.xlsx')
summary_df



MODEL: localization_no_pretrain

Fold 1/3 | test_lmi_index=0 | test=patient043/s0141lre | generator=['patient043/s0144lre', 'patient043/s0278lre']
train: (165, 1000, 12) [56 56 29 24]
val  : (26, 1000, 12) [8 8 4 6]
test : (42, 1000, 12) [16 16  9  1]
Epoch 001 | train_loss=0.8420 val_loss=0.8036 val_acc=0.2308 val_macro_f1=0.0938
Epoch 005 | train_loss=0.2349 val_loss=0.7743 val_acc=0.4231 val_macro_f1=0.2745
Epoch 010 | train_loss=0.0969 val_loss=0.3191 val_acc=0.6538 val_macro_f1=0.6591
Epoch 015 | train_loss=0.0759 val_loss=0.3386 val_acc=0.6538 val_macro_f1=0.6920
Epoch 020 | train_loss=0.0301 val_loss=0.3806 val_acc=0.6923 val_macro_f1=0.7363
Epoch 025 | train_loss=0.0388 val_loss=0.3437 val_acc=0.7692 val_macro_f1=0.7914
Epoch 030 | train_loss=0.0278 val_loss=0.3315 val_acc=0.7308 val_macro_f1=0.7643
Epoch 035 | train_loss=0.0346 val_loss=0.3405 val_acc=0.7692 val_macro_f1=0.7914
Epoch 040 | train_loss=0.0423 val_loss=0.3339 val_acc=0.7692 val_macro_f1=0.7931
Epoch 045 | train_

/tmp/ipykernel_87610/2094086319.py:43: RuntimeWarning: weights_only=True failed for local checkpoint best_model_localization_evidence.pth: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct])` or the `torch.serialization.safe_globals([numpy._core.multiarray._recons

Epoch 005 | train_loss=1.1674 val_loss=1.1701 val_acc=0.5385 val_macro_f1=0.5065
Epoch 010 | train_loss=0.4875 val_loss=0.5633 val_acc=0.5769 val_macro_f1=0.5446
Epoch 015 | train_loss=0.4686 val_loss=0.4670 val_acc=0.6538 val_macro_f1=0.6627
Epoch 020 | train_loss=0.3994 val_loss=0.4087 val_acc=0.6538 val_macro_f1=0.6627
Epoch 025 | train_loss=0.3787 val_loss=0.3722 val_acc=0.6154 val_macro_f1=0.5794
Epoch 030 | train_loss=0.3974 val_loss=0.3603 val_acc=0.5769 val_macro_f1=0.5509
Epoch 035 | train_loss=0.3391 val_loss=0.3550 val_acc=0.6538 val_macro_f1=0.6421
Epoch 040 | train_loss=0.3282 val_loss=0.3609 val_acc=0.6154 val_macro_f1=0.6129
Epoch 045 | train_loss=0.3941 val_loss=0.3522 val_acc=0.6154 val_macro_f1=0.6129
Epoch 050 | train_loss=0.3049 val_loss=0.3720 val_acc=0.5769 val_macro_f1=0.5926
Epoch 055 | train_loss=0.2853 val_loss=0.3667 val_acc=0.5385 val_macro_f1=0.5316
Epoch 060 | train_loss=0.4018 val_loss=0.3481 val_acc=0.6538 val_macro_f1=0.6627
Epoch 065 | train_loss=0.357

/tmp/ipykernel_87610/2094086319.py:43: RuntimeWarning: weights_only=True failed for local checkpoint best_model_localization_evidence.pth: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct])` or the `torch.serialization.safe_globals([numpy._core.multiarray._recons

Epoch 005 | train_loss=1.2119 val_loss=1.1243 val_acc=0.5000 val_macro_f1=0.4750
Epoch 010 | train_loss=0.4891 val_loss=0.4989 val_acc=0.5769 val_macro_f1=0.5381
Epoch 015 | train_loss=0.3847 val_loss=0.4012 val_acc=0.6154 val_macro_f1=0.6205
Epoch 020 | train_loss=0.2958 val_loss=0.3218 val_acc=0.6538 val_macro_f1=0.6627
Epoch 025 | train_loss=0.3278 val_loss=0.2884 val_acc=0.6154 val_macro_f1=0.6129
Epoch 030 | train_loss=0.2914 val_loss=0.3023 val_acc=0.5769 val_macro_f1=0.5727
Epoch 035 | train_loss=0.3184 val_loss=0.2722 val_acc=0.6538 val_macro_f1=0.6421
Epoch 040 | train_loss=0.2909 val_loss=0.2818 val_acc=0.6154 val_macro_f1=0.6342
Epoch 045 | train_loss=0.2869 val_loss=0.2736 val_acc=0.6154 val_macro_f1=0.6129
Epoch 050 | train_loss=0.3467 val_loss=0.2724 val_acc=0.6538 val_macro_f1=0.6852
Epoch 055 | train_loss=0.2934 val_loss=0.2649 val_acc=0.6154 val_macro_f1=0.6129
Epoch 060 | train_loss=0.3618 val_loss=0.2701 val_acc=0.6154 val_macro_f1=0.6129
Epoch 065 | train_loss=0.361

...

/tmp/ipykernel_87610/2094086319.py:43: RuntimeWarning: weights_only=True failed for local checkpoint best_model_localization_evidence.pth: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct])` or the `torch.serialization.safe_globals([numpy._core.multiarray._recons

Epoch 001 | train_loss=1.3154 val_loss=0.4573 val_acc=0.6538 val_macro_f1=0.6796
Epoch 005 | train_loss=0.0410 val_loss=0.2390 val_acc=0.8077 val_macro_f1=0.8105
Epoch 010 | train_loss=0.0126 val_loss=0.2569 val_acc=0.8077 val_macro_f1=0.8125
Epoch 015 | train_loss=0.0041 val_loss=0.2693 val_acc=0.8077 val_macro_f1=0.8125
Epoch 020 | train_loss=0.0017 val_loss=0.2481 val_acc=0.8846 val_macro_f1=0.8889
Epoch 025 | train_loss=0.0019 val_loss=0.2558 val_acc=0.8846 val_macro_f1=0.8889
Epoch 030 | train_loss=0.0011 val_loss=0.2607 val_acc=0.8462 val_macro_f1=0.8577
Epoch 035 | train_loss=0.0013 val_loss=0.2467 val_acc=0.8846 val_macro_f1=0.8889
Epoch 040 | train_loss=0.0015 val_loss=0.2513 val_acc=0.8846 val_macro_f1=0.8889
Epoch 045 | train_loss=0.0012 val_loss=0.2376 val_acc=0.8846 val_macro_f1=0.8889
Epoch 050 | train_loss=0.0024 val_loss=0.2352 val_acc=0.8462 val_macro_f1=0.8577
Epoch 055 | train_loss=0.0022 val_loss=0.2407 val_acc=0.8846 val_macro_f1=0.8889
Epoch 060 | train_loss=0.001

/tmp/ipykernel_87610/2094086319.py:43: RuntimeWarning: weights_only=True failed for local checkpoint best_model_localization_evidence.pth: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct])` or the `torch.serialization.safe_globals([numpy._core.multiarray._recons

Epoch 001 | train_loss=1.2370 val_loss=0.2528 val_acc=0.7692 val_macro_f1=0.7806
Epoch 005 | train_loss=0.0237 val_loss=0.2673 val_acc=0.8846 val_macro_f1=0.8865
Epoch 010 | train_loss=0.0124 val_loss=0.2389 val_acc=0.9231 val_macro_f1=0.9198
Epoch 015 | train_loss=0.0051 val_loss=0.2469 val_acc=0.8462 val_macro_f1=0.8221
Epoch 020 | train_loss=0.0009 val_loss=0.2375 val_acc=0.7692 val_macro_f1=0.7585
Epoch 025 | train_loss=0.0010 val_loss=0.3138 val_acc=0.7692 val_macro_f1=0.7585
Epoch 030 | train_loss=0.0009 val_loss=0.2649 val_acc=0.8462 val_macro_f1=0.8221
Epoch 035 | train_loss=0.0010 val_loss=0.2775 val_acc=0.8462 val_macro_f1=0.8221
Epoch 040 | train_loss=0.0016 val_loss=0.2761 val_acc=0.8462 val_macro_f1=0.8221
Epoch 045 | train_loss=0.0008 val_loss=0.3089 val_acc=0.8462 val_macro_f1=0.8221
Epoch 050 | train_loss=0.0007 val_loss=0.2725 val_acc=0.8462 val_macro_f1=0.8221
Epoch 055 | train_loss=0.0006 val_loss=0.3023 val_acc=0.8462 val_macro_f1=0.8221
Epoch 060 | train_loss=0.001

,model_name,fold,test_lmi_index,test_lmi_record,generator_lmi_records,best_epoch,best_selection_metric,best_selection_score,pretrained_loaded,freeze_mode,...,total_params,checkpoint_path,val_accuracy,val_balanced_accuracy,val_macro_f1,val_weighted_f1,test_accuracy,test_balanced_accuracy,test_macro_f1,test_weighted_f1
0,localization_no_pretrain,1,0,patient043/s0141lre,patient043/s0144lre;patient043/s0278lre,61,macro_f1,0.821429,False,full,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.807692,0.81250,0.821429,0.802198,0.952381,0.968750,0.968750,0.952381
1,localization_no_pretrain,2,1,patient043/s0144lre,patient043/s0141lre;patient043/s0278lre,26,macro_f1,0.905882,False,full,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.884615,0.90625,0.905882,0.884163,0.880952,0.909722,0.915560,0.881133
2,localization_no_pretrain,3,2,patient043/s0278lre,patient043/s0141lre;patient043/s0144lre,38,macro_f1,0.919841,False,full,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.923077,0.90625,0.919841,0.923321,0.857143,0.659722,0.659257,0.847720
3,localization_frozen_backbone,1,0,patient043/s0141lre,patient043/s0144lre;patient043/s0278lre,15,macro_f1,0.662698,True,frozen_backbone,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.653846,0.71875,0.662698,0.617827,0.428571,0.612847,0.422222,0.466667
4,localization_frozen_backbone,2,1,patient043/s0144lre,patient043/s0141lre;patient043/s0278lre,72,macro_f1,0.714286,True,frozen_backbone,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.692308,0.75000,0.714286,0.679121,0.452381,0.628472,0.451739,0.509160
5,localization_frozen_backbone,3,2,patient043/s0278lre,patient043/s0141lre;patient043/s0144lre,23,macro_f1,0.662698,True,frozen_backbone,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.653846,0.71875,0.662698,0.617827,0.428571,0.612847,0.422222,0.466667
6,localization_partial_finetune,1,0,patient043/s0141lre,patient043/s0144lre;patient043/s0278lre,23,macro_f1,0.919841,True,partial,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.923077,0.90625,0.919841,0.923321,0.904762,0.937500,0.929699,0.901647
7,localization_partial_finetune,2,1,patient043/s0144lre,patient043/s0141lre;patient043/s0278lre,45,macro_f1,0.949580,True,partial,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.961538,0.93750,0.949580,0.959922,0.928571,0.953125,0.953079,0.928502
8,localization_partial_finetune,3,2,patient043/s0278lre,patient043/s0141lre;patient043/s0144lre,11,macro_f1,0.919841,True,partial,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.923077,0.90625,0.919841,0.923321,0.904762,0.703125,0.689551,0.892850
9,localization_full_finetune,1,0,patient043/s0141lre,patient043/s0144lre;patient043/s0278lre,16,macro_f1,0.919841,True,full,...,951341,/home/nugee/code-program/code-thesis/hibah/myo...,0.923077,0.90625,0.919841,0.923321,0.928571,0.953125,0.947024,0.928046


## 7. Catatan Output

Output tersimpan di `crossval_comparison/localization_leave_one_lmi_3fold_gan/`. File `leave_one_lmi_protocol.csv` mencatat indeks LMI test dan dua record generator pada setiap fold.